In [1]:
import json
import optuna
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from pathlib import Path
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [10]:
IMAGE_EMB_PATH  = '../checkpoints/image_embeddings.pt'
TEXT_EMB_PATH   = '../checkpoints/text_embeddings.pt'
MLP_SAVE        = '../checkpoints/best_multimodal_mlp.pt'
LABEL_MAP_PATH  = Path('../data/label_map.json')
EXCLUDED_SUFFIX = 'leaf'
BATCH_SIZE      = 32

torch.manual_seed(42)
torch.cuda.manual_seed(42)

In [11]:
class MultimodalMLP(nn.Module):
    """
    Fuses image and text embeddings and classifies plant disease.
    """

    def __init__(self, image_dim=768, text_dim=768, num_classes=None, dropout=0.3):
        super().__init__()
        if num_classes is None:
            raise ValueError('num_classes must be provided.')

        fused_dim = image_dim + text_dim  # 1536

        self.mlp = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes),
        )

    def forward(self, image_emb, text_emb):
        fused = torch.cat([image_emb, text_emb], dim=1)  # (B, 1536)
        return self.mlp(fused)

In [12]:
def load_embeddings():
    """Load image and text embeddings."""
    image_data = torch.load(IMAGE_EMB_PATH)
    img_train_embeddings = image_data['train_embeddings']
    img_train_labels     = image_data['train_labels']
    img_test_embeddings  = image_data['test_embeddings']
    img_test_labels      = image_data['test_labels']

    text_data = torch.load(TEXT_EMB_PATH)
    txt_train_embeddings = text_data['train_features']
    txt_train_labels     = text_data['train_labels']
    txt_test_embeddings  = text_data['test_features']
    txt_test_labels      = text_data['test_labels']

    print(f'Image train embeddings shape:  {img_train_embeddings.shape}')
    print(f'Text  train embeddings shape:  {txt_train_embeddings.shape}')
    print(f'Image test  embeddings shape:  {img_test_embeddings.shape}')
    print(f'Text  test  embeddings shape:  {txt_test_embeddings.shape}')
    print(f'Unique image classes: {img_train_labels.unique().shape[0]}')
    print(f'Unique text  classes: {txt_train_labels.unique().shape[0]}')

    return (img_train_embeddings, img_train_labels,
            img_test_embeddings,  img_test_labels,
            txt_train_embeddings, txt_train_labels,
            txt_test_embeddings,  txt_test_labels)


def load_filtered_label_info():
    """Return the class IDs to keep after excluding labels that end with 'leaf'."""
    with LABEL_MAP_PATH.open(encoding='utf-8') as f:
        label_map = json.load(f)

    ordered_labels = sorted(label_map.items(), key=lambda item: item[1])
    excluded = [(name, idx) for name, idx in ordered_labels if name.endswith(EXCLUDED_SUFFIX)]
    kept     = [(name, idx) for name, idx in ordered_labels if not name.endswith(EXCLUDED_SUFFIX)]

    if not kept:
        raise ValueError(f"No classes remain after excluding labels ending with '{EXCLUDED_SUFFIX}'.")

    old_to_new = {old_idx: new_idx for new_idx, (_, old_idx) in enumerate(kept)}

    print(f"Excluding {len(excluded)} classes ending with '{EXCLUDED_SUFFIX}':")
    print(', '.join(name for name, _ in excluded))
    print(f'Keeping {len(kept)} classes for training/testing.')

    return {
        'excluded':    excluded,
        'kept':        kept,
        'old_to_new':  old_to_new,
        'num_classes': len(kept),
    }


def filter_and_remap_embeddings(embeddings, labels, old_to_new):
    """Drop excluded labels and remap the remaining labels to 0..N-1."""
    remap = torch.full((max(old_to_new) + 1,), -1, dtype=torch.long)
    for old_idx, new_idx in old_to_new.items():
        remap[old_idx] = new_idx

    mapped_labels = remap[labels.long()]
    keep_mask     = mapped_labels >= 0

    return embeddings[keep_mask], mapped_labels[keep_mask]


def align_embeddings(img_emb, img_lbl, txt_emb, txt_lbl, num_classes):
    """
    Align image and text embeddings by class label.
    Keeps all image samples; samples text embeddings with replacement
    to match the image count per class.
    """
    aligned_img, aligned_txt, aligned_lbl = [], [], []

    for class_idx in range(num_classes):
        img_mask  = (img_lbl == class_idx)
        txt_mask  = (txt_lbl == class_idx)
        img_class = img_emb[img_mask]
        txt_class = txt_emb[txt_mask]

        if len(img_class) == 0 or len(txt_class) == 0:
            continue

        n = len(img_class)
        repeat_idx = torch.randint(0, len(txt_class), (n,))
        aligned_img.append(img_class)
        aligned_txt.append(txt_class[repeat_idx])
        aligned_lbl.append(torch.full((n,), class_idx, dtype=torch.long))

    if not aligned_img:
        raise ValueError('No aligned classes found after filtering/remapping embeddings.')

    return (torch.cat(aligned_img),
            torch.cat(aligned_txt),
            torch.cat(aligned_lbl))


def build_loaders(train_img, train_txt, train_lbl,
                  test_img,  test_txt,  test_lbl):
    train_dataset = TensorDataset(train_img, train_txt, train_lbl)
    test_dataset  = TensorDataset(test_img,  test_txt,  test_lbl)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader

In [ ]:
label_info = load_filtered_label_info()

(img_train_emb, img_train_lbl,
 img_test_emb,  img_test_lbl,
 txt_train_emb, txt_train_lbl,
 txt_test_emb,  txt_test_lbl) = load_embeddings()

img_train_emb, img_train_lbl = filter_and_remap_embeddings(img_train_emb, img_train_lbl, label_info['old_to_new'])
img_test_emb,  img_test_lbl  = filter_and_remap_embeddings(img_test_emb,  img_test_lbl,  label_info['old_to_new'])
txt_train_emb, txt_train_lbl = filter_and_remap_embeddings(txt_train_emb, txt_train_lbl, label_info['old_to_new'])
txt_test_emb,  txt_test_lbl  = filter_and_remap_embeddings(txt_test_emb,  txt_test_lbl,  label_info['old_to_new'])

print('\nFiltered embeddings:')
print(f'Image train: {img_train_emb.shape}  Text train: {txt_train_emb.shape}')
print(f'Image test:  {img_test_emb.shape}   Text test:  {txt_test_emb.shape}')

print('\nAligning embeddings...')
train_img, train_txt, train_lbl = align_embeddings(
    img_train_emb, img_train_lbl, txt_train_emb, txt_train_lbl, label_info['num_classes'])
test_img, test_txt, test_lbl = align_embeddings(
    img_test_emb, img_test_lbl, txt_test_emb, txt_test_lbl, label_info['num_classes'])

print(f'Aligned train: img={train_img.shape}, txt={train_txt.shape}, lbl={train_lbl.shape}')
print(f'Aligned test:  img={test_img.shape},  txt={test_txt.shape},  lbl={test_lbl.shape}')

train_loader, test_loader = build_loaders(
    train_img, train_txt, train_lbl,
    test_img,  test_txt,  test_lbl)

print('\nData ready.')

Excluding 0 classes ending with 'leaf':

Keeping 59 classes for training/testing.
Image train embeddings shape:  torch.Size([8560, 768])
Text  train embeddings shape:  torch.Size([2360, 768])
Image test  embeddings shape:  torch.Size([2140, 768])
Text  test  embeddings shape:  torch.Size([590, 768])
Unique image classes: 59
Unique text  classes: 59

Filtered embeddings:
Image train: torch.Size([8560, 768])  Text train: torch.Size([2360, 768])
Image test:  torch.Size([2140, 768])   Text test:  torch.Size([590, 768])

Aligning embeddings...
Aligned train: img=torch.Size([8560, 768]), txt=torch.Size([8560, 768]), lbl=torch.Size([8560])
Aligned test:  img=torch.Size([2140, 768]),  txt=torch.Size([2140, 768]),  lbl=torch.Size([2140])

Data ready.


In [14]:
def objective(trial):
    # ── Search space ──────────────────────────────────────────
    lr      = trial.suggest_float('lr',      1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1,  0.5)
    epochs  = trial.suggest_int(  'epochs',  20,   80,   step=10)
    # ─────────────────────────────────────────────────────────

    model = MultimodalMLP(
        num_classes=label_info['num_classes'],
        dropout=dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2
    )

    for epoch in range(1, epochs + 1):
        model.train()
        for img_emb, txt_emb, labels in train_loader:
            img_emb = img_emb.to(DEVICE)
            txt_emb = txt_emb.to(DEVICE)
            labels  = labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(img_emb, txt_emb), labels)
            loss.backward()
            optimizer.step()
        scheduler.step(epoch)

        # Report intermediate value for pruning
        if epoch % 10 == 0:
            model.eval()
            preds_list, labels_list = [], []
            with torch.no_grad():
                for img_emb, txt_emb, labels in test_loader:
                    preds = model(img_emb.to(DEVICE), txt_emb.to(DEVICE)).argmax(1).cpu()
                    preds_list.extend(preds.numpy())
                    labels_list.extend(labels.numpy())
            interim_f1 = f1_score(labels_list, preds_list, average='macro', zero_division=0)
            trial.report(interim_f1, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    # ── Final eval ────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for img_emb, txt_emb, labels in test_loader:
            preds = model(img_emb.to(DEVICE), txt_emb.to(DEVICE)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    return f1_score(all_labels, all_preds, average='macro', zero_division=0)

In [16]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True,
)

[I 2026-04-10 17:13:42,273] A new study created in memory with name: no-name-01f0be17-2166-45de-9c0a-fa67eb5d6f29


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-10 17:15:20,851] Trial 0 finished with value: 0.8247122316236382 and parameters: {'lr': 0.0005611516415334506, 'dropout': 0.4802857225639665, 'epochs': 70}. Best is trial 0 with value: 0.8247122316236382.
[I 2026-04-10 17:15:58,113] Trial 1 finished with value: 0.8271502557358846 and parameters: {'lr': 0.0015751320499779737, 'dropout': 0.1624074561769746, 'epochs': 30}. Best is trial 1 with value: 0.8271502557358846.
[I 2026-04-10 17:17:44,891] Trial 2 finished with value: 0.821117375096154 and parameters: {'lr': 0.00013066739238053285, 'dropout': 0.4464704583099741, 'epochs': 60}. Best is trial 1 with value: 0.8271502557358846.
[I 2026-04-10 17:19:40,423] Trial 3 finished with value: 0.8248677867229366 and parameters: {'lr': 0.0026070247583707684, 'dropout': 0.10823379771832098, 'epochs': 80}. Best is trial 1 with value: 0.8271502557358846.
[I 2026-04-10 17:20:16,831] Trial 4 finished with value: 0.815307773540145 and parameters: {'lr': 0.004622589001020831, 'dropout': 0.18

In [17]:
print('Best trial:')
best = study.best_trial
print(f'  Macro F1 : {best.value:.4f}')
print(f'  lr       : {best.params["lr"]:.6f}')
print(f'  dropout  : {best.params["dropout"]:.3f}')
print(f'  epochs   : {best.params["epochs"]}')

print('\nTop 5 trials:')
df = study.trials_dataframe().sort_values('value', ascending=False)
print(df[['number', 'value', 'params_lr', 'params_dropout', 'params_epochs']].head())

Best trial:
  Macro F1 : 0.8333
  lr       : 0.001419
  dropout  : 0.254
  epochs   : 40

Top 5 trials:
    number     value  params_lr  params_dropout  params_epochs
17      17  0.833339   0.001419        0.254265             40
48      48  0.832097   0.004233        0.161689             60
70      70  0.830909   0.002981        0.209307             60
98      98  0.830609   0.000952        0.186244             60
60      60  0.830599   0.004256        0.307468             40


In [18]:
best_lr      = best.params['lr']
best_dropout = best.params['dropout']
best_epochs  = best.params['epochs']

print(f'Retraining with lr={best_lr:.6f}, dropout={best_dropout:.3f}, epochs={best_epochs}')

model = MultimodalMLP(
    num_classes=label_info['num_classes'],
    dropout=best_dropout
).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2
)

best_val_f1 = 0.0

for epoch in tqdm(range(1, best_epochs + 1), desc='Final training'):
    model.train()
    train_loss    = 0.0
    train_correct = train_total = 0

    for img_emb, txt_emb, labels in train_loader:
        img_emb = img_emb.to(DEVICE)
        txt_emb = txt_emb.to(DEVICE)
        labels  = labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(img_emb, txt_emb)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    scheduler.step(epoch)

    # Val F1 every epoch to save the best checkpoint
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for img_emb, txt_emb, labels in test_loader:
            preds = model(img_emb.to(DEVICE), txt_emb.to(DEVICE)).argmax(1).cpu()
            preds_list.extend(preds.numpy())
            labels_list.extend(labels.numpy())

    val_f1    = f1_score(labels_list, preds_list, average='macro', zero_division=0)
    train_acc = 100 * train_correct / train_total

    print(f'Epoch {epoch:>3}/{best_epochs}  '
          f'Loss: {train_loss / len(train_loader):.4f}  '
          f'Train Acc: {train_acc:.2f}%  '
          f'Val F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), MLP_SAVE)
        print(f'  ✓ Best model saved (F1={best_val_f1:.4f})')

print(f'\nDone — best Val F1: {best_val_f1:.4f}')

Retraining with lr=0.001419, dropout=0.254, epochs=40


Final training:   2%|▎         | 1/40 [00:02<01:33,  2.41s/it]

Epoch   1/40  Loss: 1.0235  Train Acc: 91.71%  Val F1: 0.7896
  ✓ Best model saved (F1=0.7896)


Final training:   5%|▌         | 2/40 [00:04<01:16,  2.00s/it]

Epoch   2/40  Loss: 0.8361  Train Acc: 96.69%  Val F1: 0.8140
  ✓ Best model saved (F1=0.8140)


...

Epoch  38/40  Loss: 0.7363  Train Acc: 99.84%  Val F1: 0.8288


Final training:  98%|█████████▊| 39/40 [01:13<00:02,  2.03s/it]

Epoch  39/40  Loss: 0.7376  Train Acc: 99.71%  Val F1: 0.8310


Final training: 100%|██████████| 40/40 [01:14<00:00,  1.87s/it]

Epoch  40/40  Loss: 0.7367  Train Acc: 99.80%  Val F1: 0.8303

Done — best Val F1: 0.8335


In [19]:
model.load_state_dict(torch.load(MLP_SAVE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for img_emb, txt_emb, labels in tqdm(test_loader, desc='Evaluating'):
        preds = model(img_emb.to(DEVICE), txt_emb.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

acc       = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
precision = precision_score(all_labels, all_preds, average='macro', zero_division=0) * 100
recall    = recall_score(all_labels,    all_preds, average='macro', zero_division=0) * 100
f1        = f1_score(all_labels,        all_preds, average='macro', zero_division=0) * 100

print(f'Acc:       {acc:.2f}%')
print(f'Precision: {precision:.2f}%')
print(f'Recall:    {recall:.2f}%')
print(f'F1:        {f1:.2f}%')

Evaluating: 100%|██████████| 67/67 [00:00<00:00, 266.92it/s]

Acc:       84.86%
Precision: 84.30%
Recall:    83.92%
F1:        83.35%
